Goal: able to find dependency between the property and the associated numerical value

In [1]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification
from TorchCRF import CRF
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [3]:
import datasets

In [4]:
label_map = {
    "O": 0,
    "B-PROPERTY": 1, #主语
    "I-PROPERTY": 2,
    "B-YEAR": 3, #year
    "I-YEAR": 4, 
    "B-TIME": 5, #quarter, month, 
    "I-TIME": 6,
    "B-UNIT": 7, #currency, dollar sign, etc. optional, some value may not have a unit
    "I-UNIT": 8,
    "B-VALUE": 9, #number only
    "I-VALUE": 10, 
    "B-MULTIPLIER": 11,
    "I-MULTIPLIER": 12,
    "B-CHANGE": 13, #increase, decrease, etc. for dependency parsing, to be implemented
    "I-CHANGE": 14,
    "B-CHANGE_UNIT": 15, #%, basis point, etc
    "I-CHANGE_UNIT": 16,
    "B-COMPANY": 17, #if we want to compare different companies
    "I-COMPANY": 18
}

In [144]:
train_sentences = [
    "GMV in 2023 was $ 19.2 billion",
    "Revenue in 2021 was $ 4.38 billion",
    "Revenue in 2022 was $ 5.56 billion",
    "Revenue in 2023 was $ 7.68 billion",
    "Profit in 2021 was $ 24 million",
    "Profit in 2022 was $ 372 million",
    "Profit in 2023 was $ 1.10 billion",
    "Adjusted profit in 2021 was $ 769.6 million",
    "Adjusted profit in 2022 was $ 788.1 million",
    "Adjusted profit in 2023 was $ 1.46 billion"
]

train_labels = [
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
]

train_sentences.extend([
    "Tesla 's revenue in 2023 was $ 81.5 billion",
    "Amazon 's net profit in 2022 was $ 33.3 billion",
    "Apple 's operating income in 2024 was $ 34.2 billion",
    "Google 's advertising revenue in 2023 was $ 280 billion",
    "Microsoft 's cloud revenue in 2022 was $ 72 billion",
    "Facebook 's profit margin in 2023 was 25 %",
    "Netflix 's subscriber growth in 2023 was 12 %",
    "Goldman Sachs 's investment banking revenue in 2021 was $ 14.5 billion",
    "JP Morgan 's total assets in 2023 were $ 3.9 trillion",
    "Berkshire Hathaway 's total revenue in 2023 was $ 302 billion"
])

train_labels.extend([
    ["B-COMPANY", "O", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
])

train_sentences.extend([
    "Goodme 's profit for the year was 24.0 million dollars , 372.0 million dollars and 1,096.4 million dollars in 2021 , 2022 and 2023 , respectively.",
    "Goodme 's adjusted profit was 769.6 million dollars , 788.1 million dollars and 1,459.0 million dollars in 2021 , 2022 and 2023 , respectively.",
    "The revenue in 2021 was 4.38 billion dollars",
    "The revenue in 2022 was 5.56 billion dollars",
    "The revenue in 2023 was 7.68 billion dollars",
    "Profit in 2021 was 24 million dollars",
    "Profit in 2022 was 372 million dollars",
    "Profit in 2023 was 1.10 billion dollars",
    "Adjusted profit in 2021 was 769.6 million dollars",
    "Adjusted profit in 2022 was 788.1 million dollars",
    "Adjusted profit in 2023 was 1.46 billion dollars"
])

train_labels.extend([
    ["B-COMPANY", "O", "B-PROPERTY", "O", "O", "O", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O","B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY",  "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O","B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"]
])


Preprocess:
aligns tokenized words with assigned labels 

In [145]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [146]:
def align_labels(sentence, word_labels):
    words = sentence.split()
    tokens = []
    aligned_labels = []

    word_idx = 0  # of original label

    for word in words:
        sub_tokens = tokenizer.tokenize(word)
        tokens.extend(sub_tokens)

        first_label = word_labels[word_idx]

        # If it's the first word of an entity, keep it as "B-"
        # If it's an "I-" label originally, it should stay "I-"
        sub_labels = [first_label] + [
            "I-" + first_label[2:] if first_label.startswith("B-") else first_label
            for _ in range(len(sub_tokens) - 1)
        ]

        aligned_labels.extend(sub_labels)
        word_idx += 1

    return tokens, aligned_labels

In [147]:
test_sentence = "Net income for 2021 was 4.5 billion dollars"
test_labels = ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"]

tokens, adjusted_labels = align_labels(test_sentence, test_labels)

print(f"Original Sentence: {test_sentence}")
print(f"Tokenized Output: {tokens}")
print(f"Aligned Labels: {adjusted_labels}")

Original Sentence: Net income for 2021 was 4.5 billion dollars
Tokenized Output: ['net', 'income', 'for', '2021', 'was', '4', '.', '5', 'billion', 'dollars']
Aligned Labels: ['B-PROPERTY', 'I-PROPERTY', 'O', 'B-YEAR', 'O', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER', 'B-UNIT']


In [149]:
for i, sentence in enumerate(train_sentences):
    tokens, adjusted_labels = align_labels(sentence, train_labels[i])
    print(f"Sentence {i}: {sentence}")
    print(f"Tokenized: {tokens}")
    print(f"Aligned Labels: {adjusted_labels}")
    print("-" * 40)

Sentence 0: GMV in 2023 was $ 19.2 billion
Tokenized: ['gm', '##v', 'in', '202', '##3', 'was', '$', '19', '.', '2', 'billion']
Aligned Labels: ['B-PROPERTY', 'I-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 1: Revenue in 2021 was $ 4.38 billion
Tokenized: ['revenue', 'in', '2021', 'was', '$', '4', '.', '38', 'billion']
Aligned Labels: ['B-PROPERTY', 'O', 'B-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 2: Revenue in 2022 was $ 5.56 billion
Tokenized: ['revenue', 'in', '202', '##2', 'was', '$', '5', '.', '56', 'billion']
Aligned Labels: ['B-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 3: Revenue in 2023 was $ 7.68 billion
Tokenized: ['revenue', 'in', '202', '##3', 'was', '$', '7', '.', '68', 'billion']
Align

In [150]:
def tokenize_and_align_labels(sentences, labels, max_length=50):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for i, (sentence, word_labels) in enumerate(zip(sentences, labels)):
        tokens, aligned_labels = align_labels(sentence, word_labels)

        # [CLS] and [SEP], denoting the start and end of a sentence
        tokens = ["[CLS]"] + tokens + ["[SEP]"]
        aligned_labels = ["[CLS]"] + aligned_labels + ["[SEP]"] 

        # Convert tokens to input IDs
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)

        # Convert aligned labels to numerical values using `label_map`
        label_ids = [label_map.get(lbl, 0) for lbl in aligned_labels]

        # Pad sequences to max_length
        padding_length = max_length - len(input_ids)
        if padding_length > 0:
            input_ids += [0] * padding_length  # Pad input IDs with 0 (BERT's padding)
            attention_mask += [0] * padding_length  # Pad attention mask with 0
            label_ids += [-100] * padding_length  # Pad labels with -100 to ignore them in loss

        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)

        # Debugging Output
        # print(f"\nSentence {i}: {sentence}")
        # print(f"Tokenized: {tokens}")
        # print(f"Aligned Labels: {aligned_labels}")
        # print(f"Input IDs: {input_ids}")
        # print(f"Attention Mask: {attention_mask}")
        # print(f"Label IDs: {label_ids}")
        # print("-" * 60)

    return tokenized_inputs


In [151]:
train_encodings = tokenize_and_align_labels(train_sentences, train_labels)

In [152]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} 

In [153]:
train_dataset = NERDataset(train_encodings)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [154]:
class BertCRF(torch.nn.Module):
    def __init__(self, num_labels):
        super(BertCRF, self).__init__()
        self.bert = BertForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, return_dict=False)
        emissions = outputs[0]

        if labels is not None:
            labels = labels.clone()  # Avoid modifying original tensor
            labels[labels == -100] = 0  # Replace -100 with 'O'

            # Compute CRF loss
            loss = -self.crf(emissions, labels, mask=attention_mask.bool(), reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=attention_mask.bool())


In [155]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertCRF(num_labels=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [156]:
def train_model(num_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss = model(input_ids, attention_mask, labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dataloader):.4f}")

In [157]:
train_model(num_epochs=10)

Epoch 1 Loss: 36.5176
Epoch 2 Loss: 19.3455
Epoch 3 Loss: 10.8696
Epoch 4 Loss: 5.8808
Epoch 5 Loss: 3.6731
Epoch 6 Loss: 2.4868
Epoch 7 Loss: 1.7283
Epoch 8 Loss: 1.3105
Epoch 9 Loss: 0.8902
Epoch 10 Loss: 0.6569


In [158]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt")
    inputs.pop("token_type_ids", None)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        predictions = model(**inputs)

    predicted_labels = [list(label_map.keys())[pred] for pred in predictions[0]]
    tokens = tokenizer.tokenize(tokenizer.decode(inputs["input_ids"][0]))

    return list(zip(tokens, predicted_labels))

In [160]:
test_text = "net income increased to 20 million in 2024"
print(predict(test_text))

test_text2  = "Amazon's adjusted profit was 769.6 million dollars, 788.1 million dollars and 1,459.0 million dollars in 2021, 2022 and 2023, respectively."

print(predict(test_text2))

[('[CLS]', 'O'), ('net', 'B-PROPERTY'), ('income', 'I-PROPERTY'), ('increased', 'O'), ('to', 'O'), ('20', 'B-VALUE'), ('million', 'B-MULTIPLIER'), ('in', 'O'), ('202', 'B-YEAR'), ('##4', 'I-YEAR'), ('[SEP]', 'O')]
[('[CLS]', 'O'), ('amazon', 'B-COMPANY'), ("'", 'O'), ('s', 'O'), ('adjusted', 'B-PROPERTY'), ('profit', 'I-PROPERTY'), ('was', 'O'), ('76', 'B-VALUE'), ('##9', 'I-VALUE'), ('.', 'I-VALUE'), ('6', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), (',', 'O'), ('78', 'B-VALUE'), ('##8', 'I-VALUE'), ('.', 'I-VALUE'), ('1', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('and', 'O'), ('1', 'B-VALUE'), (',', 'I-VALUE'), ('45', 'I-VALUE'), ('##9', 'I-VALUE'), ('.', 'I-VALUE'), ('0', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('in', 'O'), ('2021', 'B-YEAR'), (',', 'O'), ('202', 'B-YEAR'), ('##2', 'I-YEAR'), ('and', 'O'), ('202', 'B-YEAR'), ('##3', 'I-YEAR'), (',', 'O'), ('respectively', 'O'), ('.', 'O'), ('[SEP]', 'O')]


In [174]:
def extract_relationships(predicted_entities, sentence):
    """
    Extract structured financial relationships using BERT-CRF labels + spaCy dependency parsing.

    Returns: List of (company, financial_property, year, monetary_value)
    """
    # doc = nlp(sentence)  # Process sentence with spaCy

    relationships = []
    last_company = None
    last_year = None
    last_property = None

    entity_dict = {
        "COMPANY": [],
        "PROPERTY": [],
        "YEAR": [],
        "VALUE": [],
        "MULTIPLIER": [],
        "UNIT": [],
    }

    for token, label in predicted_entities:
        if label.startswith("B-COMPANY"):
            entity_dict["COMPANY"].append(token)
        elif label.startswith("B-PROPERTY"):
            entity_dict["PROPERTY"].append(token)
        elif label.startswith("I-PROPERTY") and entity_dict["PROPERTY"]:
            entity_dict["PROPERTY"][-1] += " " + token.replace("##", "") 
        elif label.startswith("B-YEAR"):
            entity_dict["YEAR"].append(token)
        elif label.startswith("I-YEAR") and entity_dict["YEAR"]:
            entity_dict["YEAR"][-1] += token.replace("##", "")
        elif label.startswith("B-VALUE"):
            entity_dict["VALUE"].append(token)
        elif label.startswith("I-VALUE") and entity_dict["VALUE"]:
            entity_dict["VALUE"][-1] += token.replace("##", "")
        elif label.startswith("B-MULTIPLIER"):
            entity_dict["MULTIPLIER"].append(token)
        elif label.startswith("B-UNIT"):
            entity_dict["UNIT"].append(token)

    for i in range(max(len(entity_dict["COMPANY"]), len(entity_dict["PROPERTY"]), len(entity_dict["VALUE"]))):
        
        company = entity_dict["COMPANY"][i] if i < len(entity_dict["COMPANY"]) else last_company
        property_ = entity_dict["PROPERTY"][i] if i < len(entity_dict["PROPERTY"]) else last_property
        year = entity_dict["YEAR"][i] if i < len(entity_dict["YEAR"]) else last_year
        if i < len(entity_dict["VALUE"]):
            value = entity_dict["VALUE"][i]
            value += " " + entity_dict["MULTIPLIER"][i]
            value += " " + entity_dict["UNIT"][i]
        else:
            value = None

        if year:
            last_year = year  # Update last known year
        if property_:
            last_property = property_
        if company:
            last_company = company

        if company and property_ and value:
            relationships.append((company, property_, year, value))

    return relationships

In [175]:
def process_sentence(sentence):
    """Extract structured financial data using BERT-CRF and dependency parsing."""
    predicted_entities = predict(sentence)  # Step 1: Run BERT-CRF for labeling
    print(predicted_entities)
    relationships = extract_relationships(predicted_entities, sentence)  # Step 2: Use spaCy dependency parsing

    company_data = {}

    for company, financial_property, year, monetary_value in relationships:
        year = year if year else "Unknown Year"

        if company not in company_data:
            company_data[company] = {}
            
        if financial_property not in company_data[company]:
            company_data[company][financial_property] = {}

        if year not in company_data[company][financial_property]:
            company_data[company][financial_property][year] = {}
        company_data[company][financial_property][year] = monetary_value

    # # Debugging Output
    print(f"\nSentence: {sentence}")
    # print(f"\nExtracted Relationships: {relationships}")
    # print(f"\nStructured Output: {company_data}")

    return company_data


In [180]:
sentence = "In 2023, Apple’s total revenue grew to 200 billion dollars, while Microsoft reported total revenue of 180 billion dollars. Google’s net profit in 2022 was 50 billion dollars. Amazon’s operating expenses in 2021 totaled 150 billion dollars."
process_sentence(sentence)

[('[CLS]', 'O'), ('in', 'O'), ('202', 'B-YEAR'), ('##3', 'I-YEAR'), (',', 'O'), ('apple', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('total', 'B-PROPERTY'), ('revenue', 'I-PROPERTY'), ('grew', 'O'), ('to', 'O'), ('200', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), (',', 'O'), ('while', 'O'), ('microsoft', 'B-COMPANY'), ('reported', 'O'), ('total', 'B-PROPERTY'), ('revenue', 'I-PROPERTY'), ('of', 'O'), ('180', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'O'), ('google', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('net', 'B-PROPERTY'), ('profit', 'I-PROPERTY'), ('in', 'O'), ('202', 'B-YEAR'), ('##2', 'I-YEAR'), ('was', 'O'), ('50', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'O'), ('amazon', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('operating', 'B-PROPERTY'), ('expenses', 'I-PROPERTY'), ('in', 'O'), ('2021', 'B-YEAR'), ('totaled', 'O'), ('150', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'O'), ('[SEP]',

{'apple': {'total revenue': {'2023': '200 billion dollars'}},
 'microsoft': {'total revenue': {'2022': '180 billion dollars'}},
 'google': {'net profit': {'2021': '50 billion dollars'}},
 'amazon': {'operating expenses': {'2021': '150 billion dollars'}}}

In [179]:
sentence = "Amazon's adjusted profit was 769.6 million dollars, 788.1 million dollars and 1,459.0 million dollars in 2021, 2022 and 2023, respectively."
process_sentence(sentence)

[('[CLS]', 'O'), ('amazon', 'B-COMPANY'), ("'", 'O'), ('s', 'O'), ('adjusted', 'B-PROPERTY'), ('profit', 'I-PROPERTY'), ('was', 'O'), ('76', 'B-VALUE'), ('##9', 'I-VALUE'), ('.', 'I-VALUE'), ('6', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), (',', 'O'), ('78', 'B-VALUE'), ('##8', 'I-VALUE'), ('.', 'I-VALUE'), ('1', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('and', 'O'), ('1', 'B-VALUE'), (',', 'I-VALUE'), ('45', 'I-VALUE'), ('##9', 'I-VALUE'), ('.', 'I-VALUE'), ('0', 'I-VALUE'), ('million', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('in', 'O'), ('2021', 'B-YEAR'), (',', 'O'), ('202', 'B-YEAR'), ('##2', 'I-YEAR'), ('and', 'O'), ('202', 'B-YEAR'), ('##3', 'I-YEAR'), (',', 'O'), ('respectively', 'O'), ('.', 'O'), ('[SEP]', 'O')]

Sentence: Amazon's adjusted profit was 769.6 million dollars, 788.1 million dollars and 1,459.0 million dollars in 2021, 2022 and 2023, respectively.


{'amazon': {'adjusted profit': {'2021': '769.6 million dollars',
   '2022': '788.1 million dollars',
   '2023': '1,459.0 million dollars'}}}